In [ ]:

import pyomo.environ as pyo
import pyomo.mpec as mpec
import pyomo.opt as pyoopt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
import os
from tabulate import tabulate
import warnings
import pathlib
from typing import List, Tuple
import math
import re
from collections import defaultdict


T = 24 
J = [0,1,2,3] 
J_weights = {0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}



S = ['s1', 's2'] 

States_s1 = ['idle', 'cg', 'dg', 'cs2', 'ds2', 'cgs2', 'dgs2']
States_s2 = ['idle', 'cg', 'dg', 'cs1', 'ds1', 'cgs1', 'dgs1']




demand_data=pd.read_csv('demand_data.csv'  )
demand_curve = {}
for j in J:
    demand_curve[j] = (demand_data.iloc[:, j].to_numpy() * 0.12).tolist()  # Scale down for testing
    
W = [f'w{i}' for i in range(1, 5)]  
wind_data = {}  
np.random.seed(42)  
df = pd.read_excel('para_generation_typical_days.xlsx')
time_col = df.columns[0]          
asset_col = df.columns[2]         
power_col = "Volume" # 

real_wind_names = df[asset_col].unique().tolist()

reverse_mapping = dict(zip(W, real_wind_names))
for j in J:  
    for w in W:  
        real_wind_name = reverse_mapping[w]
        turbine_data = df[df[asset_col] == real_wind_name].reset_index(drop=True)
        start_row = j * T
        end_row = (j + 1) * T

        real = turbine_data.loc[start_row:end_row-1, power_col].tolist()
        real = [val * 1.5 for val in real]  

        forecast = [ val * (0.8 + 0.6 * np.random.rand()) for val in real ]
        wind_data[(j, w)] = (0.0, forecast, real)


G = [f'g{i}' for i in range(1, 9)]   

DA_CAPACITY_FACTOR = 0.8  
RT_CAPACITY_FACTOR = 0.1 
RESERVE_CAPACITY_FACTOR = 0.1  
RESERVE_PRICE_FACTOR = 0.5  #
RT_PRICE_FACTOR = 1  #


base_gen_params = {
    'g1': (20, 300),
    'g2': (30, 300),
    'g3': (40, 300),
    'g4': (60, 300),
    'g5': (90, 300),
    'g6': (120, 300),
    'g7': (150, 400),
    'g8': (300, 400)
    # 'g9': (990, 270),
    # 'g10': (1000, 290)
}


gas_base_params = {}
for unit, (price, capacity) in base_gen_params.items():
    gas_base_params[unit] = (
        price,                                    # p_g_E_DA
        capacity * DA_CAPACITY_FACTOR,           # q_g_DA_max
        capacity * RT_CAPACITY_FACTOR,           # q_g_RT_max
        price * RESERVE_PRICE_FACTOR,            # p_g_RU_DA
        capacity * RESERVE_CAPACITY_FACTOR,      # q_g_RU_max
        price * RESERVE_PRICE_FACTOR,            # p_g_RD_DA
        capacity * RESERVE_CAPACITY_FACTOR,      # q_g_RD_max
        price * RT_PRICE_FACTOR                  # p_g_E_RT
    )



day_adjust = {
    0: {'price': 1.00, 'cap': 1.00},
    1: {'price': 1.00, 'cap': 1.00},
    2: {'price': 1.00, 'cap': 1.00},
    3: {'price': 1.00, 'cap': 1.00}
}

gas_data = {}
for j in J:
    price_factor = day_adjust[j]['price']
    cap_factor = day_adjust[j]['cap']
    for g in G:
        p_E, q_E_DA,q_E_RT, p_RU, q_RU, p_RD, q_RD, p_E_RT = gas_base_params[g]
        gas_data[(j, g)] = (
            p_E * price_factor,
            q_E_DA * cap_factor,
            q_E_RT * cap_factor,
            p_RU * price_factor,
            q_RU * cap_factor,
            p_RD * price_factor,
            q_RD * cap_factor,
            p_E_RT * price_factor,
        )



params = {
    'gamma': J_weights,
    'E_max': {'s1': 500.0, 's2': 500.0},
    'E_min': {'s1': 0.0, 's2': 0.0},
    'soc_min':0.05,
    'soc_max':0.95,
    'soc_ini':0.50,
    'eta_ch': {'s1': 0.95, 's2': 0.95},
    'eta_dch': {'s1': 0.95, 's2': 0.95},
    'c_s': {'s1': 0, 's2': 0},
    'threshold': {'s1': 0, 's2': 0},
    'C_inv': {'s1': 250000/7/365, 's2': 320000/7/365},  

    'r1': 0.2,        
    'r2': 5.0,


    'p_E_limit':200,
    'p_R_limit':200,

    'Q_D': {(j, t): demand_curve[j][t] for j in J for t in range(T)},
    'Q_RU_req': {(j, t): demand_curve[j][t] * 0.05 + sum(wind_data[(j, w)][1][t] for w in W) * 0.03
        for j in J for t in range(T)},
    'Q_RD_req': {(j, t): demand_curve[j][t] * 0.05 + sum(wind_data[(j, w)][1][t] for w in W) * 0.03
        for j in J for t in range(T)},

    # 风电 (Dict Key: (j, w, t))
    'p_w_E': {(j, w, t): wind_data[(j, w)][0] for j in J for w in W for t in range(T)},
    'q_w_E': {(j, w, t): wind_data[(j, w)][1][t] for j in J for w in W for t in range(T)},
    'q_w_real': {(j, w, t): wind_data[(j, w)][2][t] for j in J for w in W for t in range(T)},

    # 火电 (Dict Key: (j, g, t) 或 (j, g))
    'p_g_E_DA': {(j, g, t): gas_data[(j, g)][0] for j in J for g in G for t in range(T)},
    'q_g_DA': {(j, g): gas_data[(j, g)][1] for j in J for g in G},
    'q_g_RT': {(j, g): gas_data[(j, g)][2] for j in J for g in G},
    'p_g_RU_DA': {(j, g, t): gas_data[(j, g)][3] for j in J for g in G for t in range(T)},
    'q_g_RU_max': {(j, g): gas_data[(j, g)][4] for j in J for g in G},
    'p_g_RD_DA': {(j, g, t): gas_data[(j, g)][5] for j in J for g in G for t in range(T)},
    'q_g_RD_max': {(j, g): gas_data[(j, g)][6] for j in J for g in G},
    'p_g_E_RT': {(j, g, t): gas_data[(j, g)][7] for j in J for g in G for t in range(T)},
}

In [ ]:

# =============================================================================
# 2. Benders  (Master / Auxiliary / Subproblem)
# =============================================================================
BIG_M = 10000.0
DUAL_UB = 10000.0

def _bounded_nonneg(idx_sets, ub):
    return pyo.Var(*idx_sets, domain=pyo.NonNegativeReals, bounds=(0, ub))

def build_day_model(j, E_fixed=None, fix_values=None, fix_binaries=None, is_subproblem=False):
    m = pyo.ConcreteModel()
    m.J = pyo.Set(initialize=[j])
    m.T = pyo.Set(initialize=range(T))
    m.S = pyo.Set(initialize=S)
    m.W = pyo.Set(initialize=W)
    m.G = pyo.Set(initialize=G)
    m.States1 = pyo.Set(initialize=States_s1)
    m.States2 = pyo.Set(initialize=States_s2)


    m.E_s_cap = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.omega_s1 = pyo.Var(m.J, m.T, m.States1, domain=pyo.Binary)
    m.omega_s2 = pyo.Var(m.J, m.T, m.States2, domain=pyo.Binary)
    m.E_s_t = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)


    m.p_s_ch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_E_limit']))
    m.p_s_dch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_E_limit']))
    m.p_s_RU_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_R_limit']))
    m.p_s_RD_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_R_limit']))
    m.p_s_ch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_E_limit']))
    m.p_s_dch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals, bounds=(0, params['p_E_limit']))


    m.q_s_ch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.q_s_dch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.q_s_RU_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.q_s_RD_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.q_s_ch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.q_s_dch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)


    m.q_trans_1to2 = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.q_trans_2to1 = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_trans_1to2 = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_trans_2to1 = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)


    m.bar_q_s_ch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_dch_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_RU_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_RD_DA = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_w_E_DA = pyo.Var(m.J, m.W, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_E_DA = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_RU_DA = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_RD_DA = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)

    m.lambda_E_DA = pyo.Var(m.J, m.T, domain=pyo.Reals, bounds=(-DUAL_UB, DUAL_UB))
    m.lambda_RU_DA = pyo.Var(m.J, m.T, domain=pyo.Reals, bounds=(-DUAL_UB, DUAL_UB))
    m.lambda_RD_DA = pyo.Var(m.J, m.T, domain=pyo.Reals, bounds=(-DUAL_UB, DUAL_UB))


    m.bar_q_s_ch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_dch_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_RU_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_s_RD_RT = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_E_RT = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_RU_RT = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.bar_q_g_RD_RT = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)

    m.lambda_RT = pyo.Var(m.J, m.T, domain=pyo.Reals, bounds=(-DUAL_UB, DUAL_UB))
    m.dQ = pyo.Var(m.J, m.T, domain=pyo.Reals)
    m.phi_lambda_dQ = pyo.Var(m.J, m.T, domain=pyo.Reals)


    m.mu_w_E_up = _bounded_nonneg((m.J, m.W, m.T), DUAL_UB)
    m.mu_w_E_low = _bounded_nonneg((m.J, m.W, m.T), DUAL_UB)
    m.mu_g_E_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.mu_g_E_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.mu_g_RU_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.mu_g_RU_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.mu_g_RD_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.mu_g_RD_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)

    m.mu_s_ch_DA_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_ch_DA_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_dch_DA_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_dch_DA_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_RU_DA_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_RU_DA_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_RD_DA_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.mu_s_RD_DA_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)

    m.nu_g_E_RT_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.nu_g_E_RT_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.nu_g_RU_RT_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.nu_g_RU_RT_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.nu_g_RD_RT_up = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)
    m.nu_g_RD_RT_low = _bounded_nonneg((m.J, m.G, m.T), DUAL_UB)

    m.nu_s_ch_RT_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_ch_RT_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_dch_RT_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_dch_RT_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_RU_RT_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_RU_RT_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_RD_RT_up = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)
    m.nu_s_RD_RT_low = _bounded_nonneg((m.J, m.S, m.T), DUAL_UB)

    m.slack_w_E_up = pyo.Var(m.J, m.W, m.T, domain=pyo.NonNegativeReals)
    m.slack_g_E_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_g_RU_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_g_RD_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_ch_DA_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_dch_DA_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_RU_DA_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_RD_DA_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)

    m.slack_g_RU_RT_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_g_RD_RT_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_g_E_RT_up = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_RU_RT_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_RD_RT_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_ch_RT_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)
    m.slack_s_dch_RT_up = pyo.Var(m.J, m.S, m.T, domain=pyo.NonNegativeReals)


    m.phi_g_RU = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)
    m.phi_g_RD = pyo.Var(m.J, m.G, m.T, domain=pyo.NonNegativeReals)


    m.slack_DA_E = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.slack_DA_RU = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.slack_DA_RD = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)
    m.slack_RT = pyo.Var(m.J, m.T, domain=pyo.NonNegativeReals)


    if not is_subproblem:
        m.z_pairs = pyo.VarList(domain=pyo.Binary)

    m.constraints = pyo.ConstraintList()


    if E_fixed is not None:
        if is_subproblem:
            m.fix_E = pyo.ConstraintList()
            m.fix_E_map = {}
            for s in m.S:
                con = m.fix_E.add(m.E_s_cap[s] == E_fixed[s])
                m.fix_E_map[s] = con
        else:
            for s in m.S:
                m.E_s_cap[s].fix(E_fixed[s])


    for jj in m.J:
        for t in m.T:
            m.constraints.add(sum(m.omega_s1[jj, t, s] for s in States_s1) == 1)
            m.constraints.add(sum(m.omega_s2[jj, t, s] for s in States_s2) == 1)
            m.constraints.add(m.omega_s1[jj, t, 'ds2'] + m.omega_s1[jj, t, 'dgs2'] == m.omega_s2[jj, t, 'cs1'] + m.omega_s2[jj, t, 'cgs1'])
            m.constraints.add(m.omega_s2[jj, t, 'ds1'] + m.omega_s2[jj, t, 'dgs1'] == m.omega_s1[jj, t, 'cs2'] + m.omega_s1[jj, t, 'cgs2'])

            E_prev1 = m.E_s_t[jj, 's1', t-1] if t > 0 else m.E_s_cap['s1'] * params['soc_ini']
            in1 = m.bar_q_s_ch_DA[jj, 's1', t] + m.bar_q_trans_2to1[jj, t] + m.bar_q_s_RD_RT[jj, 's1', t] + m.bar_q_s_ch_RT[jj, 's1', t]
            out1 = m.bar_q_s_dch_DA[jj, 's1', t] + m.bar_q_trans_1to2[jj, t] + m.bar_q_s_RU_RT[jj, 's1', t] + m.bar_q_s_dch_RT[jj, 's1', t]
            m.constraints.add(m.E_s_t[jj, 's1', t] == E_prev1 + params['eta_ch']['s1'] * in1 - out1 / params['eta_dch']['s1'])

            E_prev2 = m.E_s_t[jj, 's2', t-1] if t > 0 else m.E_s_cap['s2'] * params['soc_ini']
            in2 = m.bar_q_s_ch_DA[jj, 's2', t] + m.bar_q_trans_1to2[jj, t] + m.bar_q_s_RD_RT[jj, 's2', t] + m.bar_q_s_ch_RT[jj, 's2', t]
            out2 = m.bar_q_s_dch_DA[jj, 's2', t] + m.bar_q_trans_2to1[jj, t] + m.bar_q_s_RU_RT[jj, 's2', t] + m.bar_q_s_dch_RT[jj, 's2', t]
            m.constraints.add(m.E_s_t[jj, 's2', t] == E_prev2 + params['eta_ch']['s2'] * in2 - out2 / params['eta_dch']['s2'])

            for s in m.S:
                m.constraints.add(m.E_s_t[jj, s, t] >= params['soc_min'] * m.E_s_cap[s])
                m.constraints.add(m.E_s_t[jj, s, t] <= params['soc_max'] * m.E_s_cap[s])

            om_t12 = m.omega_s1[jj, t, 'ds2'] + m.omega_s1[jj, t, 'dgs2']
            m.constraints.add(m.q_trans_1to2[jj, t] <= om_t12 * m.E_s_cap['s1'] * params['r1'])
            m.constraints.add(m.q_trans_1to2[jj, t] <= om_t12 * m.E_s_cap['s2'] * params['r2'])

            om_t21 = m.omega_s2[jj, t, 'ds1'] + m.omega_s2[jj, t, 'dgs1']
            m.constraints.add(m.q_trans_2to1[jj, t] <= om_t21 * m.E_s_cap['s1'] * params['r1'])
            m.constraints.add(m.q_trans_2to1[jj, t] <= om_t21 * m.E_s_cap['s2'] * params['r2'])

            for s in m.S:
                P_r = m.E_s_cap['s1'] * params['r1'] if s == 's1' else m.E_s_cap['s2'] * params['r2']
                om_dch = m.omega_s1[jj, t, 'dg'] + m.omega_s1[jj, t, 'dgs2'] if s == 's1' else m.omega_s2[jj, t, 'dg'] + m.omega_s2[jj, t, 'dgs1']
                m.constraints.add(m.q_s_dch_DA[jj, s, t] <= om_dch * P_r)
                m.constraints.add(m.q_s_dch_RT[jj, s, t] <= om_dch * P_r)
                m.constraints.add(m.q_s_RU_DA[jj, s, t] <= om_dch * P_r)

                om_dch2 = m.omega_s1[jj, t, 'dg'] + m.omega_s1[jj, t, 'ds2'] + m.omega_s1[jj, t, 'dgs2'] if s == 's1' else m.omega_s2[jj, t, 'dg'] + m.omega_s2[jj, t, 'ds1'] + m.omega_s2[jj, t, 'dgs1']
                if s == 's1':
                    m.constraints.add(m.q_s_dch_DA[jj, s, t] + m.q_trans_1to2[jj, t] + m.q_s_RU_DA[jj, s, t] + m.q_s_dch_RT[jj, s, t] <= om_dch2 * P_r)
                else:
                    m.constraints.add(m.q_s_dch_DA[jj, s, t] + m.q_trans_2to1[jj, t] + m.q_s_RU_DA[jj, s, t] + m.q_s_dch_RT[jj, s, t] <= om_dch2 * P_r)

                om_ch = m.omega_s1[jj, t, 'cg'] + m.omega_s1[jj, t, 'cgs2'] if s == 's1' else m.omega_s2[jj, t, 'cg'] + m.omega_s2[jj, t, 'cgs1']
                m.constraints.add(m.q_s_ch_DA[jj, s, t] <= om_ch * P_r)
                m.constraints.add(m.q_s_ch_RT[jj, s, t] <= om_ch * P_r)
                m.constraints.add(m.q_s_RD_DA[jj, s, t] <= om_ch * P_r)

                omch2 = m.omega_s1[jj, t, 'cg'] + m.omega_s1[jj, t, 'cs2'] + m.omega_s1[jj, t, 'cgs2'] if s == 's1' else m.omega_s2[jj, t, 'cg'] + m.omega_s2[jj, t, 'cs1'] + m.omega_s2[jj, t, 'cgs1']
                if s == 's1':
                    m.constraints.add(m.q_s_ch_DA[jj, s, t] + m.q_trans_2to1[jj, t] + m.q_s_RD_DA[jj, s, t] + m.q_s_ch_RT[jj, s, t] <= omch2 * P_r)
                else:
                    m.constraints.add(m.q_s_ch_DA[jj, s, t] + m.q_trans_1to2[jj, t] + m.q_s_RD_DA[jj, s, t] + m.q_s_ch_RT[jj, s, t] <= omch2 * P_r)

            m.constraints.add(m.bar_q_trans_1to2[jj, t] <= m.q_trans_1to2[jj, t])
            m.constraints.add(m.bar_q_trans_2to1[jj, t] <= m.q_trans_2to1[jj, t])

        for s in m.S:
            m.constraints.add(m.E_s_t[jj, s, T-1] - m.E_s_cap[s] * params['soc_ini'] >= params['threshold'][s])


    for jj in m.J:
        for t in m.T:
            for w in m.W:
                m.constraints.add(params['p_w_E'][jj, w, t] - m.lambda_E_DA[jj, t] - m.mu_w_E_low[jj, w, t] + m.mu_w_E_up[jj, w, t] == 0)
            for g in m.G:
                m.constraints.add(params['p_g_E_DA'][jj, g, t] - m.lambda_E_DA[jj, t] - m.mu_g_E_low[jj, g, t] + m.mu_g_E_up[jj, g, t] == 0)
                m.constraints.add(params['p_g_RU_DA'][jj, g, t] - m.lambda_RU_DA[jj, t] - m.mu_g_RU_low[jj, g, t] + m.mu_g_RU_up[jj, g, t] == 0)
                m.constraints.add(params['p_g_RD_DA'][jj, g, t] - m.lambda_RD_DA[jj, t] - m.mu_g_RD_low[jj, g, t] + m.mu_g_RD_up[jj, g, t] == 0)
            for s in m.S:
                m.constraints.add(m.p_s_dch_DA[jj, s, t] - m.lambda_E_DA[jj, t] - m.mu_s_dch_DA_low[jj, s, t] + m.mu_s_dch_DA_up[jj, s, t] == 0)
                m.constraints.add(-m.p_s_ch_DA[jj, s, t] + m.lambda_E_DA[jj, t] - m.mu_s_ch_DA_low[jj, s, t] + m.mu_s_ch_DA_up[jj, s, t] == 0)
                m.constraints.add(m.p_s_RU_DA[jj, s, t] - m.lambda_RU_DA[jj, t] - m.mu_s_RU_DA_low[jj, s, t] + m.mu_s_RU_DA_up[jj, s, t] == 0)
                m.constraints.add(m.p_s_RD_DA[jj, s, t] - m.lambda_RD_DA[jj, t] - m.mu_s_RD_DA_low[jj, s, t] + m.mu_s_RD_DA_up[jj, s, t] == 0)

            for g in m.G:
                m.constraints.add(params['p_g_E_DA'][jj, g, t] - m.lambda_RT[jj, t] - m.nu_g_RU_RT_low[jj, g, t] + m.nu_g_RU_RT_up[jj, g, t] == 0)
                m.constraints.add(-params['p_g_E_DA'][jj, g, t] + m.lambda_RT[jj, t] - m.nu_g_RD_RT_low[jj, g, t] + m.nu_g_RD_RT_up[jj, g, t] == 0)
                m.constraints.add(params['p_g_E_RT'][jj, g, t] - m.lambda_RT[jj, t] - m.nu_g_E_RT_low[jj, g, t] + m.nu_g_E_RT_up[jj, g, t] == 0)
            for s in m.S:
                m.constraints.add(m.p_s_dch_DA[jj, s, t] - m.lambda_RT[jj, t] - m.nu_s_RU_RT_low[jj, s, t] + m.nu_s_RU_RT_up[jj, s, t] == 0)
                m.constraints.add(-m.p_s_ch_DA[jj, s, t] + m.lambda_RT[jj, t] - m.nu_s_RD_RT_low[jj, s, t] + m.nu_s_RD_RT_up[jj, s, t] == 0)
                m.constraints.add(m.p_s_dch_RT[jj, s, t] - m.lambda_RT[jj, t] - m.nu_s_dch_RT_low[jj, s, t] + m.nu_s_dch_RT_up[jj, s, t] == 0)
                m.constraints.add(-m.p_s_ch_RT[jj, s, t] + m.lambda_RT[jj, t] - m.nu_s_ch_RT_low[jj, s, t] + m.nu_s_ch_RT_up[jj, s, t] == 0)

            m.constraints.add(sum(m.bar_q_w_E_DA[jj, w, t] for w in m.W) + sum(m.bar_q_g_E_DA[jj, g, t] for g in m.G) + sum(m.bar_q_s_dch_DA[jj, s, t] - m.bar_q_s_ch_DA[jj, s, t] for s in m.S) + m.slack_DA_E[jj, t] == params['Q_D'][jj, t])
            m.constraints.add(sum(m.bar_q_s_RU_DA[jj, s, t] for s in m.S) + sum(m.bar_q_g_RU_DA[jj, g, t] for g in m.G) + m.slack_DA_RU[jj, t] == params['Q_RU_req'][jj, t])
            m.constraints.add(sum(m.bar_q_s_RD_DA[jj, s, t] for s in m.S) + sum(m.bar_q_g_RD_DA[jj, g, t] for g in m.G) + m.slack_DA_RD[jj, t] == params['Q_RD_req'][jj, t])

            m.constraints.add(m.dQ[jj, t] == sum(m.bar_q_w_E_DA[jj, w, t] - params['q_w_real'][jj, w, t] for w in m.W))
            m.constraints.add(sum(m.bar_q_g_RU_RT[jj, g, t] - m.bar_q_g_RD_RT[jj, g, t] for g in m.G) +
                              sum(m.bar_q_s_RU_RT[jj, s, t] - m.bar_q_s_RD_RT[jj, s, t] for s in m.S) +
                              sum(m.bar_q_g_E_RT[jj, g, t] for g in m.G) +
                              sum(m.bar_q_s_dch_RT[jj, s, t] - m.bar_q_s_ch_RT[jj, s, t] for s in m.S) - m.dQ[jj, t] + m.slack_RT[jj, t] == 0)


    for jj in m.J:
        for t in m.T:
            for w in m.W:
                m.constraints.add(m.slack_w_E_up[jj, w, t] == params['q_w_E'][jj, w, t] - m.bar_q_w_E_DA[jj, w, t])
            for g in m.G:
                m.constraints.add(m.slack_g_E_up[jj, g, t] == params['q_g_DA'][jj, g] - m.bar_q_g_E_DA[jj, g, t])
                m.constraints.add(m.slack_g_RU_up[jj, g, t] == params['q_g_RU_max'][jj, g] - m.bar_q_g_RU_DA[jj, g, t])
                m.constraints.add(m.slack_g_RD_up[jj, g, t] == params['q_g_RD_max'][jj, g] - m.bar_q_g_RD_DA[jj, g, t])
                m.constraints.add(m.slack_g_RU_RT_up[jj, g, t] == m.bar_q_g_RU_DA[jj, g, t] - m.bar_q_g_RU_RT[jj, g, t])
                m.constraints.add(m.slack_g_RD_RT_up[jj, g, t] == m.bar_q_g_RD_DA[jj, g, t] - m.bar_q_g_RD_RT[jj, g, t])
                m.constraints.add(m.slack_g_E_RT_up[jj, g, t] == params['q_g_RT'][jj, g] - m.bar_q_g_E_RT[jj, g, t])
            for s in m.S:
                m.constraints.add(m.slack_s_ch_DA_up[jj, s, t] == m.q_s_ch_DA[jj, s, t] - m.bar_q_s_ch_DA[jj, s, t])
                m.constraints.add(m.slack_s_dch_DA_up[jj, s, t] == m.q_s_dch_DA[jj, s, t] - m.bar_q_s_dch_DA[jj, s, t])
                m.constraints.add(m.slack_s_RU_DA_up[jj, s, t] == m.q_s_RU_DA[jj, s, t] - m.bar_q_s_RU_DA[jj, s, t])
                m.constraints.add(m.slack_s_RD_DA_up[jj, s, t] == m.q_s_RD_DA[jj, s, t] - m.bar_q_s_RD_DA[jj, s, t])

                m.constraints.add(m.slack_s_RU_RT_up[jj, s, t] == m.bar_q_s_RU_DA[jj, s, t] - m.bar_q_s_RU_RT[jj, s, t])
                m.constraints.add(m.slack_s_RD_RT_up[jj, s, t] == m.bar_q_s_RD_DA[jj, s, t] - m.bar_q_s_RD_RT[jj, s, t])
                m.constraints.add(m.slack_s_ch_RT_up[jj, s, t] == m.q_s_ch_RT[jj, s, t] - m.bar_q_s_ch_RT[jj, s, t])
                m.constraints.add(m.slack_s_dch_RT_up[jj, s, t] == m.q_s_dch_RT[jj, s, t] - m.bar_q_s_dch_RT[jj, s, t])


    if not is_subproblem:
        def _add_big_m(mu, slack_or_var):
            z = m.z_pairs.add()
            m.constraints.add(mu <= BIG_M * z)
            m.constraints.add(slack_or_var <= BIG_M * (1 - z))
            return z

        for jj in m.J:
            for t in m.T:
                for w in m.W:
                    _add_big_m(m.mu_w_E_low[jj, w, t], m.bar_q_w_E_DA[jj, w, t])
                    _add_big_m(m.mu_w_E_up[jj, w, t], m.slack_w_E_up[jj, w, t])
                for g in m.G:
                    _add_big_m(m.mu_g_E_low[jj, g, t], m.bar_q_g_E_DA[jj, g, t])
                    _add_big_m(m.mu_g_E_up[jj, g, t], m.slack_g_E_up[jj, g, t])
                    _add_big_m(m.mu_g_RU_low[jj, g, t], m.bar_q_g_RU_DA[jj, g, t])
                    _add_big_m(m.mu_g_RU_up[jj, g, t], m.slack_g_RU_up[jj, g, t])
                    _add_big_m(m.mu_g_RD_low[jj, g, t], m.bar_q_g_RD_DA[jj, g, t])
                    _add_big_m(m.mu_g_RD_up[jj, g, t], m.slack_g_RD_up[jj, g, t])
                    _add_big_m(m.nu_g_RU_RT_low[jj, g, t], m.bar_q_g_RU_RT[jj, g, t])
                    _add_big_m(m.nu_g_RU_RT_up[jj, g, t], m.slack_g_RU_RT_up[jj, g, t])
                    _add_big_m(m.nu_g_RD_RT_low[jj, g, t], m.bar_q_g_RD_RT[jj, g, t])
                    _add_big_m(m.nu_g_RD_RT_up[jj, g, t], m.slack_g_RD_RT_up[jj, g, t])
                    _add_big_m(m.nu_g_E_RT_low[jj, g, t], m.bar_q_g_E_RT[jj, g, t])
                    _add_big_m(m.nu_g_E_RT_up[jj, g, t], m.slack_g_E_RT_up[jj, g, t])
                for s in m.S:
                    _add_big_m(m.mu_s_dch_DA_low[jj, s, t], m.bar_q_s_dch_DA[jj, s, t])
                    _add_big_m(m.mu_s_dch_DA_up[jj, s, t], m.slack_s_dch_DA_up[jj, s, t])
                    _add_big_m(m.mu_s_ch_DA_low[jj, s, t], m.bar_q_s_ch_DA[jj, s, t])
                    _add_big_m(m.mu_s_ch_DA_up[jj, s, t], m.slack_s_ch_DA_up[jj, s, t])
                    _add_big_m(m.mu_s_RU_DA_low[jj, s, t], m.bar_q_s_RU_DA[jj, s, t])
                    _add_big_m(m.mu_s_RU_DA_up[jj, s, t], m.slack_s_RU_DA_up[jj, s, t])
                    _add_big_m(m.mu_s_RD_DA_low[jj, s, t], m.bar_q_s_RD_DA[jj, s, t])
                    _add_big_m(m.mu_s_RD_DA_up[jj, s, t], m.slack_s_RD_DA_up[jj, s, t])
                    _add_big_m(m.nu_s_RU_RT_low[jj, s, t], m.bar_q_s_RU_RT[jj, s, t])
                    _add_big_m(m.nu_s_RU_RT_up[jj, s, t], m.slack_s_RU_RT_up[jj, s, t])
                    _add_big_m(m.nu_s_RD_RT_low[jj, s, t], m.bar_q_s_RD_RT[jj, s, t])
                    _add_big_m(m.nu_s_RD_RT_up[jj, s, t], m.slack_s_RD_RT_up[jj, s, t])
                    _add_big_m(m.nu_s_dch_RT_low[jj, s, t], m.bar_q_s_dch_RT[jj, s, t])
                    _add_big_m(m.nu_s_dch_RT_up[jj, s, t], m.slack_s_dch_RT_up[jj, s, t])
                    _add_big_m(m.nu_s_ch_RT_low[jj, s, t], m.bar_q_s_ch_RT[jj, s, t])
                    _add_big_m(m.nu_s_ch_RT_up[jj, s, t], m.slack_s_ch_RT_up[jj, s, t])


    for jj in m.J:
        for g in m.G:
            for t in m.T:
                q_ru_max = params['q_g_RU_max'][jj, g]
                q_rd_max = params['q_g_RD_max'][jj, g]
                m.constraints.add(m.phi_g_RU[jj, g, t] >= 0)
                m.constraints.add(m.phi_g_RU[jj, g, t] >= q_ru_max * m.nu_g_RU_RT_up[jj, g, t] + DUAL_UB * m.bar_q_g_RU_DA[jj, g, t] - DUAL_UB * q_ru_max)
                m.constraints.add(m.phi_g_RU[jj, g, t] <= q_ru_max * m.nu_g_RU_RT_up[jj, g, t])
                m.constraints.add(m.phi_g_RU[jj, g, t] <= DUAL_UB * m.bar_q_g_RU_DA[jj, g, t])

                m.constraints.add(m.phi_g_RD[jj, g, t] >= 0)
                m.constraints.add(m.phi_g_RD[jj, g, t] >= q_rd_max * m.nu_g_RD_RT_up[jj, g, t] + DUAL_UB * m.bar_q_g_RD_DA[jj, g, t] - DUAL_UB * q_rd_max)
                m.constraints.add(m.phi_g_RD[jj, g, t] <= q_rd_max * m.nu_g_RD_RT_up[jj, g, t])
                m.constraints.add(m.phi_g_RD[jj, g, t] <= DUAL_UB * m.bar_q_g_RD_DA[jj, g, t])

    for jj in m.J:
        for t in m.T:
            dQ_min = -sum(params['q_w_real'][jj, w, t] for w in m.W)
            dQ_max = sum(params['q_w_E'][jj, w, t] - params['q_w_real'][jj, w, t] for w in m.W)
            m.dQ[jj, t].setlb(dQ_min)
            m.dQ[jj, t].setub(dQ_max)
            xL = 0
            xU = DUAL_UB
            yL = dQ_min
            yU = dQ_max

            m.constraints.add(m.phi_lambda_dQ[jj,t] >= xL * m.dQ[jj,t] + yL * m.lambda_RT[jj,t] - xL * yL)
            m.constraints.add(m.phi_lambda_dQ[jj,t] >= xU * m.dQ[jj,t] + yU * m.lambda_RT[jj,t] - xU * yU)
            m.constraints.add(m.phi_lambda_dQ[jj,t] <= xU * m.dQ[jj,t] + yL * m.lambda_RT[jj,t] - xU * yL)
            m.constraints.add(m.phi_lambda_dQ[jj,t] <= xL * m.dQ[jj,t] + yU * m.lambda_RT[jj,t] - xL * yU)


    if fix_values is not None:
        for key, val in fix_values.items():
            var_name, idx = key
            getattr(m, var_name)[idx].fix(val)
    if fix_binaries is not None and not is_subproblem:
        for z_idx, z_val in fix_binaries.items():
            m.z_pairs[z_idx].fix(z_val)


    if is_subproblem:
        for jj in m.J:
            for t in m.T:
                Y_t = (
                    sum(params['p_w_E'][jj, w, t] * m.bar_q_w_E_DA[jj, w, t] for w in m.W)
                    + sum(
                        params['p_g_E_DA'][jj, g, t] * m.bar_q_g_E_DA[jj, g, t]
                        + params['p_g_RU_DA'][jj, g, t] * m.bar_q_g_RU_DA[jj, g, t]
                        + params['p_g_RD_DA'][jj, g, t] * m.bar_q_g_RD_DA[jj, g, t]
                        for g in m.G
                    )
                )
                X_t = (
                    -sum(m.mu_w_E_up[jj, w, t] * params['q_w_E'][jj, w, t] for w in m.W)
                    - sum(
                        m.mu_g_E_up[jj, g, t] * params['q_g_DA'][jj, g]
                        + m.mu_g_RU_up[jj, g, t] * params['q_g_RU_max'][jj, g]
                        + m.mu_g_RD_up[jj, g, t] * params['q_g_RD_max'][jj, g]
                        for g in m.G
                    )
                    + m.lambda_E_DA[jj, t] * params['Q_D'][jj, t]
                    + m.lambda_RU_DA[jj, t] * params['Q_RU_req'][jj, t]
                    + m.lambda_RD_DA[jj, t] * params['Q_RD_req'][jj, t]
                )
                lhs_da = (
                    sum(
                        m.p_s_dch_DA[jj, s, t] * m.bar_q_s_dch_DA[jj, s, t]
                        - m.p_s_ch_DA[jj, s, t] * m.bar_q_s_ch_DA[jj, s, t]
                        + m.p_s_RU_DA[jj, s, t] * m.bar_q_s_RU_DA[jj, s, t]
                        + m.p_s_RD_DA[jj, s, t] * m.bar_q_s_RD_DA[jj, s, t]
                        for s in m.S
                    )
                    + Y_t
                )
                rhs_da = (
                    -sum(
                        m.mu_s_ch_DA_up[jj, s, t] * m.q_s_ch_DA[jj, s, t]
                        + m.mu_s_dch_DA_up[jj, s, t] * m.q_s_dch_DA[jj, s, t]
                        + m.mu_s_RU_DA_up[jj, s, t] * m.q_s_RU_DA[jj, s, t]
                        + m.mu_s_RD_DA_up[jj, s, t] * m.q_s_RD_DA[jj, s, t]
                        for s in m.S
                    )
                    + X_t
                )
                m.constraints.add(lhs_da == rhs_da)

                dQ = m.dQ[jj, t]
                A_t = sum(
                    params['p_g_E_DA'][jj, g, t] * (m.bar_q_g_RU_RT[jj, g, t] - m.bar_q_g_RD_RT[jj, g, t])
                    + params['p_g_E_RT'][jj, g, t] * m.bar_q_g_E_RT[jj, g, t]
                    for g in m.G
                )
                B_t = (
                    -sum(
                        m.nu_g_E_RT_up[jj, g, t] * params['q_g_RT'][jj, g]
                        + m.phi_g_RU[jj, g, t]
                        + m.phi_g_RD[jj, g, t]
                        for g in m.G
                    )
                    + m.phi_lambda_dQ[jj, t]
                )
                lhs_rt = (
                    sum(
                        m.p_s_dch_DA[jj, s, t] * m.bar_q_s_RU_RT[jj, s, t]
                        - m.p_s_ch_DA[jj, s, t] * m.bar_q_s_RD_RT[jj, s, t]
                        for s in m.S
                    )
                    + sum(
                        m.p_s_dch_RT[jj, s, t] * m.bar_q_s_dch_RT[jj, s, t]
                        - m.p_s_ch_RT[jj, s, t] * m.bar_q_s_ch_RT[jj, s, t]
                        for s in m.S
                    )
                    + A_t
                )
                rhs_rt = (
                    -sum(
                        m.nu_s_ch_RT_up[jj, s, t] * m.q_s_ch_RT[jj, s, t]
                        + m.nu_s_dch_RT_up[jj, s, t] * m.q_s_dch_RT[jj, s, t]
                        + m.nu_s_RU_RT_up[jj, s, t] * m.bar_q_s_RU_DA[jj, s, t]
                        + m.nu_s_RD_RT_up[jj, s, t] * m.bar_q_s_RD_DA[jj, s, t]
                        for s in m.S
                    )
                    + B_t
                )
                m.constraints.add(lhs_rt == rhs_rt)


    def _profit_rule(mm):
        total_profit = 0
        slack_penalty = 1e6
        for jj in mm.J:
            primal_cost = sum(
                sum(params['p_w_E'][jj, w, t] * mm.bar_q_w_E_DA[jj, w, t] for w in mm.W) +
                sum(params['p_g_E_DA'][jj, g, t] * mm.bar_q_g_E_DA[jj, g, t] +
                    params['p_g_RU_DA'][jj, g, t] * mm.bar_q_g_RU_DA[jj, g, t] +
                    params['p_g_RD_DA'][jj, g, t] * mm.bar_q_g_RD_DA[jj, g, t] for g in mm.G)
                for t in mm.T
            )
            dual_load = sum(mm.lambda_E_DA[jj, t] * params['Q_D'][jj, t] + mm.lambda_RU_DA[jj, t] * params['Q_RU_req'][jj, t] + mm.lambda_RD_DA[jj, t] * params['Q_RD_req'][jj, t] for t in mm.T)
            dual_rent = sum(
                sum(mm.mu_w_E_up[jj, w, t] * params['q_w_E'][jj, w, t] for w in mm.W) +
                sum(mm.mu_g_E_up[jj, g, t] * params['q_g_DA'][jj, g] + mm.mu_g_RU_up[jj, g, t] * params['q_g_RU_max'][jj, g] + mm.mu_g_RD_up[jj, g, t] * params['q_g_RD_max'][jj, g] for g in mm.G)
                for t in mm.T
            )
            da_part = -primal_cost + dual_load - dual_rent

            rt_part = 0
            for t in mm.T:
                A = sum(params['p_g_E_DA'][jj, g, t] * (mm.bar_q_g_RU_RT[jj, g, t] - mm.bar_q_g_RD_RT[jj, g, t]) +
                        params['p_g_E_RT'][jj, g, t] * mm.bar_q_g_E_RT[jj, g, t] for g in mm.G)
                B = sum(mm.nu_g_E_RT_up[jj, g, t] * params['q_g_RT'][jj, g] + mm.phi_g_RU[jj, g, t] + mm.phi_g_RD[jj, g, t] for g in mm.G)
                rt_part += -(A + B - mm.phi_lambda_dQ[jj, t])

            phys_cost = sum(sum(params['c_s'][s] * (mm.bar_q_s_dch_DA[jj, s, t] + mm.bar_q_s_ch_DA[jj, s, t] + mm.bar_q_s_RU_RT[jj, s, t] + mm.bar_q_s_dch_RT[jj, s, t] + mm.bar_q_s_RD_RT[jj, s, t] + mm.bar_q_s_ch_RT[jj, s, t]) for s in mm.S) +
                            (params['c_s']['s1'] + params['c_s']['s2']) * (mm.bar_q_trans_1to2[jj, t] + mm.bar_q_trans_2to1[jj, t]) for t in mm.T)

            penalty = sum(mm.slack_DA_E[jj, t] + mm.slack_DA_RU[jj, t] + mm.slack_DA_RD[jj, t] + mm.slack_RT[jj, t] for t in mm.T)
            total_profit += params['gamma'][jj] * (da_part + rt_part - phys_cost - slack_penalty * penalty)
        return total_profit

    m.objective = pyo.Objective(rule=_profit_rule, sense=pyo.maximize)
    if is_subproblem:
        m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m

def build_master(cuts):
    m = pyo.ConcreteModel()
    m.S = pyo.Set(initialize=S)
    m.J = pyo.Set(initialize=J)
    m.E_s_cap = pyo.Var(m.S, domain=pyo.NonNegativeReals)
    m.alpha = pyo.Var(m.J, domain=pyo.Reals)
    m.constraints = pyo.ConstraintList()
    for s in m.S:
        m.constraints.add(m.E_s_cap[s] <= params['E_max'][s])
        m.constraints.add(m.E_s_cap[s] >= params['E_min'][s])
    for cut in cuts:
        j = cut['j']
        expr = cut['Pi']
        for s in m.S:
            expr += cut['pi'][s] * (m.E_s_cap[s] - cut['E_ref'][s])
        m.constraints.add(m.alpha[j] <= expr)

    m.objective = pyo.Objective(expr=(sum(m.alpha[j] for j in m.J) - sum(params['C_inv'][s] * m.E_s_cap[s] for s in m.S)), sense=pyo.maximize)
    return m

def _safe_value(v, name):
    val = pyo.value(v, exception=False)
    if val is None:
        raise ValueError(f"Var not initialized: {name}")
    return val

def _safe_binary(v, name, tol=1e-6):
    val = _safe_value(v, name)
    if abs(val) <= tol:
        return 0
    if abs(val - 1) <= tol:
        return 1
    return 1 if val >= 0.5 else 0

def _collect_fix_values(m):
    fixed = {}
    for jj in m.J:
        for t in m.T:
            for s in m.States1:
                fixed[('omega_s1', (jj, t, s))] = _safe_binary(m.omega_s1[jj, t, s], f"omega_s1[{jj},{t},{s}]")
            for s in m.States2:
                fixed[('omega_s2', (jj, t, s))] = _safe_binary(m.omega_s2[jj, t, s], f"omega_s2[{jj},{t},{s}]")
            for s in m.S:
                fixed[('p_s_ch_DA', (jj, s, t))] = _safe_value(m.p_s_ch_DA[jj, s, t], f"p_s_ch_DA[{jj},{s},{t}]")
                fixed[('p_s_dch_DA', (jj, s, t))] = _safe_value(m.p_s_dch_DA[jj, s, t], f"p_s_dch_DA[{jj},{s},{t}]")
                fixed[('p_s_RU_DA', (jj, s, t))] = _safe_value(m.p_s_RU_DA[jj, s, t], f"p_s_RU_DA[{jj},{s},{t}]")
                fixed[('p_s_RD_DA', (jj, s, t))] = _safe_value(m.p_s_RD_DA[jj, s, t], f"p_s_RD_DA[{jj},{s},{t}]")
                fixed[('p_s_ch_RT', (jj, s, t))] = _safe_value(m.p_s_ch_RT[jj, s, t], f"p_s_ch_RT[{jj},{s},{t}]")
                fixed[('p_s_dch_RT', (jj, s, t))] = _safe_value(m.p_s_dch_RT[jj, s, t], f"p_s_dch_RT[{jj},{s},{t}]")
                fixed[('mu_s_ch_DA_up', (jj, s, t))] = _safe_value(m.mu_s_ch_DA_up[jj, s, t], f"mu_s_ch_DA_up[{jj},{s},{t}]")
                fixed[('mu_s_dch_DA_up', (jj, s, t))] = _safe_value(m.mu_s_dch_DA_up[jj, s, t], f"mu_s_dch_DA_up[{jj},{s},{t}]")
                fixed[('mu_s_RU_DA_up', (jj, s, t))] = _safe_value(m.mu_s_RU_DA_up[jj, s, t], f"mu_s_RU_DA_up[{jj},{s},{t}]")
                fixed[('mu_s_RD_DA_up', (jj, s, t))] = _safe_value(m.mu_s_RD_DA_up[jj, s, t], f"mu_s_RD_DA_up[{jj},{s},{t}]")
                fixed[('nu_s_ch_RT_up', (jj, s, t))] = _safe_value(m.nu_s_ch_RT_up[jj, s, t], f"nu_s_ch_RT_up[{jj},{s},{t}]")
                fixed[('nu_s_dch_RT_up', (jj, s, t))] = _safe_value(m.nu_s_dch_RT_up[jj, s, t], f"nu_s_dch_RT_up[{jj},{s},{t}]")
                fixed[('nu_s_RU_RT_up', (jj, s, t))] = _safe_value(m.nu_s_RU_RT_up[jj, s, t], f"nu_s_RU_RT_up[{jj},{s},{t}]")
                fixed[('nu_s_RD_RT_up', (jj, s, t))] = _safe_value(m.nu_s_RD_RT_up[jj, s, t], f"nu_s_RD_RT_up[{jj},{s},{t}]")

    return fixed

def _collect_fix_binaries(m):
    return {i: pyo.value(m.z_pairs[i]) for i in range(1, len(m.z_pairs) + 1)}

# =============================================================================
# Solve
# =============================================================================
solver_master = pyo.SolverFactory('gurobi')
solver_ap = pyo.SolverFactory('gurobi')
solver_sp = pyo.SolverFactory('gurobi')
solver_ap.options['NonConvex'] = 2
solver_ap.options['MIPGap'] = 0.01
solver_master.options['MIPGap'] = 0.01

max_iter = 20
tol = 1 #%

cuts = []
E_initial = {'s1': 90, 's2': 90}
E_current=E_initial
UB = float('inf')
LB = -float('inf')
best_sp_models = {}

ub_hist = []
lb_hist = []
iter_hist = []

for k in range(1, max_iter + 1):
    if cuts:
        master = build_master(cuts)
        solver_master.solve(master, tee=False)
        E_current = {s: pyo.value(master.E_s_cap[s]) for s in master.S}
        UB = pyo.value(master.objective)
    else:
        UB = float('inf')

    total_profit = 0
    pi_values = {}
    for j in J:
        ap = build_day_model(j, E_fixed=E_current, is_subproblem=False)
        ap_results = solver_ap.solve(ap, tee=False)
        if ap_results.solver.termination_condition not in (pyoopt.TerminationCondition.optimal, pyoopt.TerminationCondition.feasible):
            raise ValueError(f"AP not solved for day {j}: {ap_results.solver.termination_condition}")
        # best_sp_models[j] = ap
        fix_values = _collect_fix_values(ap)

        sp = build_day_model(j, E_fixed=E_current, fix_values=fix_values, is_subproblem=True)
        solver_sp.solve(sp, tee=False)
        pi = {}
        for s in sp.S:
            con = sp.fix_E_map[s]
            pi[s] = sp.dual.get(con, 0.0)
        pi_values[j] = pi
        best_sp_models[j] = sp
        total_profit += pyo.value(sp.objective)
        cuts.append({'j': j, 'Pi': pyo.value(sp.objective), 'pi': pi, 'E_ref': E_current})

    inv_cost = sum(params['C_inv'][s] * E_current[s] for s in S)
    LB = total_profit - inv_cost

    iter_hist.append(k)
    ub_hist.append(UB if np.isfinite(UB) else np.nan)
    lb_hist.append(LB)
    if np.isfinite(UB):
        denom = max(abs(LB), 1e-9)
        gap_pct = abs(UB - LB) / denom * 100
        print(f"Iter {k}: UB={UB:.4f}, LB={LB:.4f}, gap={gap_pct:.4f}%, total_profit={total_profit:.4f}, inv_cost={inv_cost:.4f}")
        if gap_pct <= tol:
            print("Benders converged.")
            break
    else:
        print(f"Iter {k}: UB=inf, LB={LB:.4f} (warm start cuts)")

print("Benders finished.")

In [ ]:

# =============================================================================
# save results
# =============================================================================
import pathlib
from typing import List, Tuple
import math


def _param_dict_to_df(data: dict, index_names: List[str], j_filter=None):
    rows = []
    for k, v in data.items():
        key = k if isinstance(k, tuple) else (k,)
        if j_filter is not None and len(key) > 0 and key[0] != j_filter:
            continue
        rows.append(tuple(key) + (v,))
    if not rows:
        return pd.DataFrame(columns=index_names + ['value'])
    df = pd.DataFrame(rows, columns=index_names + ['value'])
    return df.sort_values(index_names).reset_index(drop=True)

def _var_to_df(var_obj, index_names: List[str], j_filter=None):
    rows = []
    for key in var_obj:
        key_tuple = key if isinstance(key, tuple) else (key,)
        if j_filter is not None and len(key_tuple) > 0 and key_tuple[0] != j_filter:
            continue
        try:
            val = pyo.value(var_obj[key_tuple])
        except Exception:
            val = math.nan
        rows.append(tuple(key_tuple) + (val,))
    if not rows:
        return pd.DataFrame(columns=index_names + ['value'])
    df = pd.DataFrame(rows, columns=index_names + ['value'])
    return df.sort_values(index_names).reset_index(drop=True)

def _tidy_param(name: str, data: dict, index_names: List[str], j_filter=None):
    df = _param_dict_to_df(data, index_names, j_filter)
    if df.empty:
        return df
    df.insert(0, 'item', name)
    return df

def _tidy_var(name: str, var_obj, index_names: List[str], j_filter=None):
    df = _var_to_df(var_obj, index_names, j_filter)
    if df.empty:
        return df
    df.insert(0, 'item', name)
    return df

# 参数分类
da_param_specs = [
    ('Q_D', params['Q_D'], ['j', 't']),
    ('Q_RU_req', params['Q_RU_req'], ['j', 't']),
    ('Q_RD_req', params['Q_RD_req'], ['j', 't']),
    ('p_w_E', params['p_w_E'], ['j', 'w', 't']),
    ('q_w_E', params['q_w_E'], ['j', 'w', 't']),
    ('p_g_E_DA', params['p_g_E_DA'], ['j', 'g', 't']),
    ('q_g_DA', params['q_g_DA'], ['j', 'g']),
    ('p_g_RU_DA', params['p_g_RU_DA'], ['j', 'g', 't']),
    ('q_g_RU_max', params['q_g_RU_max'], ['j', 'g']),
    ('p_g_RD_DA', params['p_g_RD_DA'], ['j', 'g', 't']),
    ('q_g_RD_max', params['q_g_RD_max'], ['j', 'g']),
]

rt_param_specs = [
    ('p_g_E_RT', params['p_g_E_RT'], ['j', 'g', 't']),
    ('q_g_RT', params['q_g_RT'], ['j', 'g']),
    ('q_w_real', params['q_w_real'], ['j', 'w', 't']),
]

def _build_var_specs(model_obj):
    da_var_specs = [
        ('lambda_E_DA', model_obj.lambda_E_DA, ['j', 't']),
        ('lambda_RU_DA', model_obj.lambda_RU_DA, ['j', 't']),
        ('lambda_RD_DA', model_obj.lambda_RD_DA, ['j', 't']),
        ('p_s_ch_DA', model_obj.p_s_ch_DA, ['j', 's', 't']),
        ('p_s_dch_DA', model_obj.p_s_dch_DA, ['j', 's', 't']),
        ('p_s_RU_DA', model_obj.p_s_RU_DA, ['j', 's', 't']),
        ('p_s_RD_DA', model_obj.p_s_RD_DA, ['j', 's', 't']),
        ('q_s_ch_DA', model_obj.q_s_ch_DA, ['j', 's', 't']),
        ('q_s_dch_DA', model_obj.q_s_dch_DA, ['j', 's', 't']),
        ('q_s_RU_DA', model_obj.q_s_RU_DA, ['j', 's', 't']),
        ('q_s_RD_DA', model_obj.q_s_RD_DA, ['j', 's', 't']),
        ('bar_q_s_ch_DA', model_obj.bar_q_s_ch_DA, ['j', 's', 't']),
        ('bar_q_s_dch_DA', model_obj.bar_q_s_dch_DA, ['j', 's', 't']),
        ('bar_q_s_RU_DA', model_obj.bar_q_s_RU_DA, ['j', 's', 't']),
        ('bar_q_s_RD_DA', model_obj.bar_q_s_RD_DA, ['j', 's', 't']),
        ('bar_q_w_E_DA', model_obj.bar_q_w_E_DA, ['j', 'w', 't']),
        ('bar_q_g_E_DA', model_obj.bar_q_g_E_DA, ['j', 'g', 't']),
        ('bar_q_g_RU_DA', model_obj.bar_q_g_RU_DA, ['j', 'g', 't']),
        ('bar_q_g_RD_DA', model_obj.bar_q_g_RD_DA, ['j', 'g', 't']),
        ('q_trans_1to2', model_obj.q_trans_1to2, ['j', 't']),
        ('q_trans_2to1', model_obj.q_trans_2to1, ['j', 't']),
        ('bar_q_trans_1to2', model_obj.bar_q_trans_1to2, ['j', 't']),
        ('bar_q_trans_2to1', model_obj.bar_q_trans_2to1, ['j', 't']),
        ('E_s_t', model_obj.E_s_t, ['j', 's', 't']),
        ('omega_s1', model_obj.omega_s1, ['j', 't', 'state']),
        ('omega_s2', model_obj.omega_s2, ['j', 't', 'state']),
    ]

    rt_var_specs = [
        ('lambda_RT', model_obj.lambda_RT, ['j', 't']),
        ('p_s_ch_RT', model_obj.p_s_ch_RT, ['j', 's', 't']),
        ('p_s_dch_RT', model_obj.p_s_dch_RT, ['j', 's', 't']),
        ('q_s_ch_RT', model_obj.q_s_ch_RT, ['j', 's', 't']),
        ('q_s_dch_RT', model_obj.q_s_dch_RT, ['j', 's', 't']),
        ('bar_q_s_ch_RT', model_obj.bar_q_s_ch_RT, ['j', 's', 't']),
        ('bar_q_s_dch_RT', model_obj.bar_q_s_dch_RT, ['j', 's', 't']),
        ('bar_q_s_RU_RT', model_obj.bar_q_s_RU_RT, ['j', 's', 't']),
        ('bar_q_s_RD_RT', model_obj.bar_q_s_RD_RT, ['j', 's', 't']),
        ('bar_q_g_E_RT', model_obj.bar_q_g_E_RT, ['j', 'g', 't']),
        ('bar_q_g_RU_RT', model_obj.bar_q_g_RU_RT, ['j', 'g', 't']),
        ('bar_q_g_RD_RT', model_obj.bar_q_g_RD_RT, ['j', 'g', 't']),
    ]

    storage_specs = [
        ('E_s_t', model_obj.E_s_t, ['j', 's', 't']),
        ('p_s_ch_DA', model_obj.p_s_ch_DA, ['j', 's', 't']),
        ('p_s_dch_DA', model_obj.p_s_dch_DA, ['j', 's', 't']),
        ('p_s_RU_DA', model_obj.p_s_RU_DA, ['j', 's', 't']),
        ('p_s_RD_DA', model_obj.p_s_RD_DA, ['j', 's', 't']),
        ('q_s_ch_DA', model_obj.q_s_ch_DA, ['j', 's', 't']),
        ('q_s_dch_DA', model_obj.q_s_dch_DA, ['j', 's', 't']),
        ('q_s_RU_DA', model_obj.q_s_RU_DA, ['j', 's', 't']),
        ('q_s_RD_DA', model_obj.q_s_RD_DA, ['j', 's', 't']),
        ('bar_q_s_ch_DA', model_obj.bar_q_s_ch_DA, ['j', 's', 't']),
        ('bar_q_s_dch_DA', model_obj.bar_q_s_dch_DA, ['j', 's', 't']),
        ('bar_q_s_RU_DA', model_obj.bar_q_s_RU_DA, ['j', 's', 't']),
        ('bar_q_s_RD_DA', model_obj.bar_q_s_RD_DA, ['j', 's', 't']),
        ('p_s_ch_RT', model_obj.p_s_ch_RT, ['j', 's', 't']),
        ('p_s_dch_RT', model_obj.p_s_dch_RT, ['j', 's', 't']),
        ('q_s_ch_RT', model_obj.q_s_ch_RT, ['j', 's', 't']),
        ('q_s_dch_RT', model_obj.q_s_dch_RT, ['j', 's', 't']),
        ('bar_q_s_ch_RT', model_obj.bar_q_s_ch_RT, ['j', 's', 't']),
        ('bar_q_s_dch_RT', model_obj.bar_q_s_dch_RT, ['j', 's', 't']),
        ('bar_q_s_RU_RT', model_obj.bar_q_s_RU_RT, ['j', 's', 't']),
        ('bar_q_s_RD_RT', model_obj.bar_q_s_RD_RT, ['j', 's', 't']),
        ('q_trans_1to2', model_obj.q_trans_1to2, ['j', 't']),
        ('q_trans_2to1', model_obj.q_trans_2to1, ['j', 't']),
        ('bar_q_trans_1to2', model_obj.bar_q_trans_1to2, ['j', 't']),
        ('bar_q_trans_2to1', model_obj.bar_q_trans_2to1, ['j', 't']),
    ]
    return da_var_specs, rt_var_specs, storage_specs

def _build_long_table(specs: List[Tuple[str, object, List[str]]], j_filter=None, is_param=False):
    frames = []
    for name, obj, idx_names in specs:
        if is_param:
            df = _tidy_param(name, obj, idx_names, j_filter)
        else:
            df = _tidy_var(name, obj, idx_names, j_filter)
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def _compute_da_balance(model_obj, j):
    rows = []
    for t in model_obj.T:
        supply = sum(pyo.value(model_obj.bar_q_w_E_DA[j, w, t]) for w in model_obj.W) + \
                 sum(pyo.value(model_obj.bar_q_g_E_DA[j, g, t]) for g in model_obj.G) + \
                 sum(pyo.value(model_obj.bar_q_s_dch_DA[j, s, t]) - pyo.value(model_obj.bar_q_s_ch_DA[j, s, t]) for s in model_obj.S)
        demand = params['Q_D'][j, t]
        ru_supply = sum(pyo.value(model_obj.bar_q_s_RU_DA[j, s, t]) for s in model_obj.S) + \
                    sum(pyo.value(model_obj.bar_q_g_RU_DA[j, g, t]) for g in model_obj.G)
        rd_supply = sum(pyo.value(model_obj.bar_q_s_RD_DA[j, s, t]) for s in model_obj.S) + \
                    sum(pyo.value(model_obj.bar_q_g_RD_DA[j, g, t]) for g in model_obj.G)
        rows.append({
            'j': j,
            't': t,
            'DA_supply': supply,
            'DA_demand': demand,
            'DA_gap': supply - demand,
            'RU_supply': ru_supply,
            'RU_req': params['Q_RU_req'][j, t],
            'RU_gap': ru_supply - params['Q_RU_req'][j, t],
            'RD_supply': rd_supply,
            'RD_req': params['Q_RD_req'][j, t],
            'RD_gap': rd_supply - params['Q_RD_req'][j, t],
        })
    return pd.DataFrame(rows)

def _compute_rt_balance(model_obj, j):
    rows = []
    for t in model_obj.T:
        imbalance = sum(pyo.value(model_obj.bar_q_g_RU_RT[j, g, t]) - pyo.value(model_obj.bar_q_g_RD_RT[j, g, t]) for g in model_obj.G) + \
                    sum(pyo.value(model_obj.bar_q_s_RU_RT[j, s, t]) - pyo.value(model_obj.bar_q_s_RD_RT[j, s, t]) for s in model_obj.S) + \
                    sum(pyo.value(model_obj.bar_q_g_E_RT[j, g, t]) for g in model_obj.G) + \
                    sum(pyo.value(model_obj.bar_q_s_dch_RT[j, s, t]) - pyo.value(model_obj.bar_q_s_ch_RT[j, s, t]) for s in model_obj.S)
        dQ = sum(pyo.value(model_obj.bar_q_w_E_DA[j, w, t]) - params['q_w_real'][j, w, t] for w in model_obj.W)
        rows.append({
            'j': j,
            't': t,
            'RT_imbalance_LHS': imbalance,
            'RT_dQ_wind': dQ,
            'RT_gap': imbalance - dQ,
        })
    return pd.DataFrame(rows)

def export_results(model_obj, base_dir='exports'):
    out_dir = pathlib.Path(base_dir)
    out_dir.mkdir(exist_ok=True)
    for j in model_obj.J:
        da_var_specs, rt_var_specs, storage_specs = _build_var_specs(model_obj)
        da_params_df = _build_long_table(da_param_specs, j_filter=j, is_param=True)
        rt_params_df = _build_long_table(rt_param_specs, j_filter=j, is_param=True)
        da_vars_df = _build_long_table(da_var_specs, j_filter=j, is_param=False)
        rt_vars_df = _build_long_table(rt_var_specs, j_filter=j, is_param=False)
        storage_df = _build_long_table(storage_specs, j_filter=j, is_param=False)
        da_balance_df = _compute_da_balance(model_obj, j)
        rt_balance_df = _compute_rt_balance(model_obj, j)

        excel_path = out_dir / f'market_export_j{j}.xlsx'
        with pd.ExcelWriter(excel_path) as writer:
            da_params_df.to_excel(writer, sheet_name='DA_params', index=False)
            da_vars_df.to_excel(writer, sheet_name='DA_vars', index=False)
            da_balance_df.to_excel(writer, sheet_name='DA_balance', index=False)
            rt_params_df.to_excel(writer, sheet_name='RT_params', index=False)
            rt_vars_df.to_excel(writer, sheet_name='RT_vars', index=False)
            rt_balance_df.to_excel(writer, sheet_name='RT_balance', index=False)
            storage_df.to_excel(writer, sheet_name='Storage', index=False)
        print(f'finished: {excel_path}')

if best_sp_models:
    for j in best_sp_models:
        export_results(best_sp_models[j])



investment_df = pd.DataFrame([
    {'storage': 's1', 'investment_MWh': E_current['s1']},
    {'storage': 's2', 'investment_MWh': E_current['s2']}
])

investment_path = pathlib.Path('exports') / 'investment_results.xlsx'
investment_df.to_excel(investment_path, index=False)
